# 실습 02 · Token·ID·Context 분석 (기본필수)

문장이 LLM에 들어가기 전에 **숫자(Token ID)로 바뀌는 과정**을 셀 하나씩 확인합니다.

### 사용 방법
1. **결과는 이미 셀 아래에 저장되어 있습니다.** 읽기만 해도 흐름을 따라갈 수 있어요.
2. 직접 돌려 보려면 `파일 → Drive에 사본 저장`으로 내 사본을 만드세요.
3. 맨 위 **⚙️ 준비 셀을 먼저 한 번 실행**한 뒤, 보고 싶은 구간의 셀을 실행하세요. 각 구간은 준비 셀만 있으면 따로 실행됩니다.
4. **✏️ 표시 셀**은 오른쪽 입력 칸의 값을 바꾸고 실행해 보세요.

| 구간 | 내용 |
|---|---|
| 0 | 실습 데이터 보기 |
| 1 | Tokenizer(단어 사전) |
| B01 | 문장 → Token·ID, 개수 세기 |
| B02 | 길이 맞추기(Padding)와 Mask |
| 실험 | 잘림(Truncation)과 모르는 단어(UNK) |
| B03 | Context 예산 계산 |
| B04 | 결과 저장 (`main.py`와 같은 결과) |

In [1]:
#@title ⚙️ 먼저 실행하세요 — 실습 공통 코드 준비
# ── 원본: 20261113/mentor/01_basic/common/support.py (그대로 포함) ──
"""강의 공통 오프라인 계산 도구: 실제 사전학습 LLM이 아닙니다.

교육용 정규표현식 토큰화·작은 벡터·가중치 계산만 제공합니다.
Transformer의 tokenizer/embedding 품질을 이 도구로 평가할 수 없습니다.
"""
from pathlib import Path
import json
import math
import re
import torch

DATA = [
    {"id":"A", "title":"실습 파일 보관", "content":"실습 파일은 개인 폴더에 보관합니다.", "category":"learning"},
    {"id":"B", "title":"서버 실행", "content":"프로젝트 폴더에서 서버를 실행합니다.", "category":"environment"},
    {"id":"C", "title":"질문 작성", "content":"오류 메시지와 시도한 작업을 함께 적습니다.", "category":"learning"},
]
SPECIAL = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]

def words(text: str) -> list[str]:
    """교육용: 한글 연속 글자/영문/숫자/문장부호 분리. 서브워드 토크나이저가 아님."""
    return re.findall(r'[가-힣]+|[A-Za-z]+|\d+|[^\w\s]', text)

class ToyTokenizer:
    """BERT tokenizer 대체가 아닌, 오프라인 개념 확인용 규칙 기반 어휘표."""
    def __init__(self, corpus):
        wordset=sorted({word for line in corpus for word in words(line)})
        self.vocab={tok:i for i,tok in enumerate(SPECIAL + wordset)}
        self.reverse={i:t for t,i in self.vocab.items()}

    def encode(self, text: str, max_length=None, padding_to=None):
        raw=words(text)
        tokens=["[BOS]"]+raw+["[EOS]"]
        truncated=False
        if max_length is not None:
            if max_length<2: raise ValueError("BOS/EOS를 포함하려면 max_length>=2")
            if len(tokens)>max_length:
                tokens=tokens[:max_length-1]+["[EOS]"]
                truncated=True
        valid=len(tokens)
        if padding_to is not None:
            if padding_to<valid: raise ValueError("padding_to must be >= valid length")
            tokens=tokens+["[PAD]"]*(padding_to-valid)
        ids=[self.vocab.get(t,self.vocab["[UNK]"]) for t in tokens]
        mask=[0 if t=="[PAD]" else 1 for t in tokens]
        return {"tokens":tokens,"input_ids":ids,"attention_mask":mask,
                "valid_length":valid,"truncated":truncated,"original_length":len(raw)+2}

    def batch(self, texts, max_length=None):
        items=[self.encode(t,max_length=max_length) for t in texts]
        max_size=max(len(i["input_ids"]) for i in items)
        return [self.encode(t,max_length=max_length,padding_to=max_size) for t in texts]

def context_budget(window, parts:dict, output_reserved:int):
    used=sum(parts.values())
    return {"window":window,"input_used":used,"output_reserved":output_reserved,
            "remaining":window-used-output_reserved,
            "fits":used+output_reserved<=window}

def cosine(a,b):
    a=torch.as_tensor(a,dtype=torch.float32)
    b=torch.as_tensor(b,dtype=torch.float32)
    return float(torch.nn.functional.cosine_similarity(a,b,dim=0))

def attention(q,k,v,causal=True):
    q=torch.as_tensor(q,dtype=torch.float32)
    k=torch.as_tensor(k,dtype=torch.float32)
    v=torch.as_tensor(v,dtype=torch.float32)
    scores=q@k.T/math.sqrt(q.shape[-1])
    mask=torch.triu(torch.ones_like(scores,dtype=torch.bool),diagonal=1)
    weights=torch.softmax(scores.masked_fill(mask,float('-inf')) if causal else scores,dim=-1)
    return scores,weights,weights@v,mask

def write_json(path, data):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')

def write_md(path,text):
    p=Path(path);p.parent.mkdir(exist_ok=True,parents=True)
    p.write_text(text,encoding='utf-8')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

# ── 노트북 전용 준비 ──
import pandas as pd
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

texts = [item["content"] for item in DATA]
tok = ToyTokenizer(texts + ["AI", "FastAPI와 PostgreSQL", "print(x) // code"])

def token_table(record):
    """encode 결과를 위치별 표로 보여 줌: token / id / mask"""
    return pd.DataFrame([record["tokens"], record["input_ids"], record["attention_mask"]],
                        index=["token", "id", "mask"])

print("준비 완료 ✅  사전 크기:", len(tok.vocab))

준비 완료 ✅  사전 크기: 30


## 0. 실습 데이터 보기
지식 문장 3건(A, B, C)입니다. 11/12에 DB에 넣었던 지식 데이터와 같은 형태입니다. Tokenizer에는 `content`만 넣습니다.

In [2]:
pd.DataFrame(DATA)

,id,title,content,category
0,A,실습 파일 보관,실습 파일은 개인 폴더에 보관합니다.,learning
1,B,서버 실행,프로젝트 폴더에서 서버를 실행합니다.,environment
2,C,질문 작성,오류 메시지와 시도한 작업을 함께 적습니다.,learning


## 1. Tokenizer(단어 사전)

### 1-1. 문장을 조각으로 자르기
한글 덩어리 / 영어 / 숫자 / 문장부호 단위로 자릅니다.

In [3]:
words("실습 파일은 개인 폴더에 보관합니다.")

['실습', '파일은', '개인', '폴더에', '보관합니다', '.']

In [4]:
words("FastAPI와 PostgreSQL")

['FastAPI', '와', 'PostgreSQL']

### 1-2. 사전 보기
조각마다 번호가 붙어 있습니다. 앞의 0~3번은 **특수 Token**입니다.
`[PAD]` 빈칸 · `[UNK]` 모르는 단어 · `[BOS]` 문장 시작 · `[EOS]` 문장 끝

In [5]:
pd.DataFrame(tok.vocab.items(), columns=["token", "id"]).set_index("id").T

id,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
token,[PAD],[UNK],[BOS],[EOS],(,),.,/,AI,FastAPI,...,실행합니다,오류,와,작업을,적습니다,파일은,폴더에,폴더에서,프로젝트,함께


## B01. 문장 → Token·ID

### B01-1. 문장 A 하나를 변환
`encode()` 결과는 딕셔너리입니다.

In [6]:
record = tok.encode(DATA[0]["content"])
record

{'tokens': ['[BOS]', '실습', '파일은', '개인', '폴더에', '보관합니다', '.', '[EOS]'],
 'input_ids': [2, 19, 25, 14, 26, 16, 6, 3],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1],
 'valid_length': 8,
 'truncated': False,
 'original_length': 8}

### B01-2. 표로 보기
위치마다 **token(조각) → id(번호) → mask(1 = 진짜 Token)** 입니다. 맨 앞뒤에 `[BOS]`, `[EOS]`가 붙은 것을 확인하세요.

In [7]:
token_table(record)

,0,1,2,3,4,5,6,7
token,[BOS],실습,파일은,개인,폴더에,보관합니다,.,[EOS]
id,2,19,25,14,26,16,6,3
mask,1,1,1,1,1,1,1,1


### B01-3. 단어 수 vs Token 수
보이는 단어는 6개인데 Token은 8개입니다. 특수 Token 2개가 더해지기 때문입니다.

In [8]:
sentence = DATA[0]["content"]
print("단어 수  :", len(words(sentence)))
print("Token 수 :", tok.encode(sentence)["valid_length"])

단어 수  : 6
Token 수 : 8


### B01-4. A/B/C 전체 처리 (`main.py`와 같은 코드)

In [9]:
rows = []
for item in DATA:
    record = tok.encode(item["content"])
    rows.append({"id": item["id"], "text": item["content"],
                 "tokens": record["tokens"], "ids": record["input_ids"],
                 "count": record["valid_length"], "special_count": 2})

pd.DataFrame(rows)[["id", "count", "tokens", "ids"]]

,id,count,tokens,ids
0,A,8,"[[BOS], 실습, 파일은, 개인, 폴더에, 보관합니다, ., [EOS]]","[2, 19, 25, 14, 26, 16, 6, 3]"
1,B,7,"[[BOS], 프로젝트, 폴더에서, 서버를, 실행합니다, ., [EOS]]","[2, 28, 27, 17, 20, 6, 3]"
2,C,9,"[[BOS], 오류, 메시지와, 시도한, 작업을, 함께, 적습니다, ., [EOS]]","[2, 21, 15, 18, 23, 29, 24, 6, 3]"


In [10]:
counts = [(r["id"], r["count"]) for r in rows]
assert counts == [("A", 8), ("B", 7), ("C", 9)], counts
print("B01 통과 ✅", counts)

B01 통과 ✅ [('A', 8), ('B', 7), ('C', 9)]


### ✏️ B01 직접 해 보기

In [11]:
#@title ✏️ 문장을 바꿔 보세요
sentence = "실습 파일은 개인 폴더에 보관합니다."  #@param {type:"string"}

my = tok.encode(sentence)
print("Token 수:", my["valid_length"], "  (사전에 없는 단어는 [UNK] = 1)")
token_table(my)

Token 수: 8   (사전에 없는 단어는 [UNK] = 1)


,0,1,2,3,4,5,6,7
token,[BOS],실습,파일은,개인,폴더에,보관합니다,.,[EOS]
id,2,19,25,14,26,16,6,3
mask,1,1,1,1,1,1,1,1


## B02. 길이 맞추기(Padding)와 Mask
여러 문장을 한 번에 처리하려면 **길이가 같아야** 합니다. 짧은 문장 뒤에 `[PAD]`(id 0)를 채웁니다.

### B02-1. 따로 변환하면 길이가 다르다

In [12]:
print("AI                  :", len(tok.encode("AI")["input_ids"]))
print("FastAPI와 PostgreSQL:", len(tok.encode("FastAPI와 PostgreSQL")["input_ids"]))

AI                  : 3
FastAPI와 PostgreSQL: 5


### B02-2. `batch()`로 함께 변환하면 길이가 맞춰진다

In [13]:
batch = tok.batch(["AI", "FastAPI와 PostgreSQL"])
token_table(batch[0])

,0,1,2,3,4
token,[BOS],AI,[EOS],[PAD],[PAD]
id,2,8,3,0,0
mask,1,1,1,0,0


In [14]:
token_table(batch[1])

,0,1,2,3,4
token,[BOS],FastAPI,와,PostgreSQL,[EOS]
id,2,9,22,10,3
mask,1,1,1,1,1


### B02-3. 전체 길이 vs 실제 Token 수
**가장 헷갈리는 부분입니다.** PAD를 채워도 실제 Token 수(`valid_length`)는 그대로이고, mask가 0인 칸이 PAD입니다.

In [15]:
pd.DataFrame([{"문장": ["AI", "FastAPI와 PostgreSQL"][i],
               "전체 길이": len(x["input_ids"]),
               "실제 Token": x["valid_length"],
               "PAD 수": x["attention_mask"].count(0)} for i, x in enumerate(batch)])

,문장,전체 길이,실제 Token,PAD 수
0,AI,5,3,2
1,FastAPI와 PostgreSQL,5,5,0


In [16]:
assert len(batch[0]["input_ids"]) == len(batch[1]["input_ids"])
assert all(len(x["input_ids"]) == len(x["attention_mask"]) for x in batch)
print("B02 통과 ✅ 맞춰진 길이 =", len(batch[0]["input_ids"]))

B02 통과 ✅ 맞춰진 길이 = 5


### ✏️ B02 직접 해 보기

In [17]:
#@title ✏️ 길이가 다른 두 문장을 넣어 보세요
first = "AI"  #@param {type:"string"}
second = "실습 파일은 개인 폴더에 보관합니다."  #@param {type:"string"}

pair = tok.batch([first, second])
display(token_table(pair[0]))
display(token_table(pair[1]))

,0,1,2,3,4,5,6,7
token,[BOS],AI,[EOS],[PAD],[PAD],[PAD],[PAD],[PAD]
id,2,8,3,0,0,0,0,0
mask,1,1,1,0,0,0,0,0


,0,1,2,3,4,5,6,7
token,[BOS],실습,파일은,개인,폴더에,보관합니다,.,[EOS]
id,2,19,25,14,26,16,6,3
mask,1,1,1,1,1,1,1,1


## 실험. 잘림(Truncation)과 모르는 단어(UNK)
과제에는 없지만 실제 Tokenizer에서도 똑같이 일어납니다. `max_length`보다 길면 뒷부분이 잘려 **모델이 아예 보지 못합니다.**

In [18]:
#@title ✏️ 잘라 낼 길이를 바꿔 보세요
sentence = "오류 메시지와 시도한 작업을 함께 적습니다."  #@param {type:"string"}
max_length = 5  #@param {type:"slider", min:2, max:12, step:1}

cut = tok.encode(sentence, max_length=max_length)
print("원래 Token 수:", cut["original_length"], " → 잘렸나?", cut["truncated"])
token_table(cut)

원래 Token 수: 9  → 잘렸나? True


,0,1,2,3,4
token,[BOS],오류,메시지와,시도한,[EOS]
id,2,21,15,18,3
mask,1,1,1,1,1


## B03. Context 예산 계산
LLM이 한 번에 다루는 Token 한도(Context Window) 안에 **입력 + 답변 공간**이 모두 들어가야 합니다.

| 항목 | 의미 |
|---|---|
| system | 시스템 지시문 |
| history | 이전 대화 |
| evidence | 찾아온 근거 문서 (11/20 RAG) |
| user | 이번 질문 |
| 출력 예약 | 답변을 쓸 공간 |

In [19]:
parts = {"system": 300, "history": 1200, "evidence": 2000, "user": 150}
budget = context_budget(5000, parts, 850)
budget

{'window': 5000,
 'input_used': 3650,
 'output_reserved': 850,
 'remaining': 500,
 'fits': True}

In [20]:
print(f"입력 {budget['input_used']} + 출력 예약 {budget['output_reserved']} "
      f"= {budget['input_used'] + budget['output_reserved']} / 한도 {budget['window']}")
print("남는 공간:", budget["remaining"])
assert budget["remaining"] == 500 and budget["fits"]
print("B03 통과 ✅")

입력 3650 + 출력 예약 850 = 4500 / 한도 5000
남는 공간: 500
B03 통과 ✅


### ✏️ B03 직접 해 보기 — 근거 문서를 늘리면?

In [21]:
#@title ✏️ 숫자를 바꿔 보세요
window = 5000  #@param {type:"integer"}
system = 300  #@param {type:"integer"}
history = 1200  #@param {type:"integer"}
evidence = 2000  #@param {type:"slider", min:0, max:5000, step:100}
user = 150  #@param {type:"integer"}
output_reserved = 850  #@param {type:"integer"}

b = context_budget(window, {"system": system, "history": history, "evidence": evidence, "user": user}, output_reserved)
print("남는 공간:", b["remaining"])
print("✅ 들어갑니다" if b["fits"] else "❌ 넘칩니다 — history나 evidence를 줄여야 합니다")

남는 공간: 500
✅ 들어갑니다


## B04. 결과 저장 (`main.py`와 같은 결과)
B01~B03 결과를 모아 `outputs/token_report.json`으로 저장합니다. 이 셀은 준비 셀만 있으면 혼자 실행됩니다.

In [22]:
rows = []
for item in DATA:
    record = tok.encode(item["content"])
    rows.append({"id": item["id"], "text": item["content"],
                 "tokens": record["tokens"], "ids": record["input_ids"],
                 "count": record["valid_length"], "special_count": 2})
batch = tok.batch(["AI", "FastAPI와 PostgreSQL"])
budget = context_budget(5000, {"system": 300, "history": 1200, "evidence": 2000, "user": 150}, 850)

result = {"tokenizer_kind": "교육용 regex·고정 어휘표 (사전학습 모델이 아님)",
          "records": rows, "batch": batch, "budget": budget, "truncation": {},
          "batch_report": [], "language_comparison": [],
          "overflow_budget": None, "policy": "예산과 잘림의 차이를 설명"}
write_json(OUT_DIR / "token_report.json", result)

print("[02]", [(x["id"], x["count"]) for x in rows], "remaining", budget["remaining"])
print("저장:", OUT_DIR / "token_report.json")

[02] [('A', 8), ('B', 7), ('C', 9)] remaining 500
저장: outputs\token_report.json


In [23]:
#@title 📥 결과 파일 내려받기 (Colab)
if IN_COLAB:
    from google.colab import files
    for name in ['token_report.json']:
        files.download(str(OUT_DIR / name))
else:
    print("Colab에서 실행하면 결과 파일이 내려받아집니다:", ['token_report.json'])

Colab에서 실행하면 결과 파일이 내려받아집니다: ['token_report.json']


---
### 정리
- **B01** 문장 → Token → ID. Token 수에는 `[BOS]`/`[EOS]`가 포함된다.
- **B02** 여러 문장을 묶으면 PAD로 길이를 맞춘다. mask 0 = PAD, 실제 Token 수는 `valid_length`.
- **실험** 한도를 넘으면 뒷부분이 잘려 모델이 보지 못한다.
- **B03** Context Window에는 입력뿐 아니라 **답변 공간**도 남겨야 한다.